In [2]:
import pandas as pd
import glob

In [ ]:
fn_ch_given_names = '../../../data/v2/chinese_names/CNKI_uniqueChineseGivenNames.csv'
path_predictions = '../../../results/v2/chinese_gender/predictions'
path_gender_tools = '../../../results/v2/chinese_gender/tool_<tool>/CNKI_uniqueChineseGivenNames_gender.csv'

In [10]:
df_names = pd.read_csv(fn_ch_given_names)
df_names.shape

(39713, 3)

In [11]:
df_names.head(2)

,givenname,pinyin,english
0,卫民,wèimín,weimin
1,丽英,lìyīng#####líyīng,liying


In [ ]:
def load_classifier(without_w=False):
    from catboost import CatBoostClassifier
    model = CatBoostClassifier()
    if not without_w:
        model.load_model('../../../results/v2/chinese_gender/comparison/classification/_mean/mean_230_1.cbm', format="cbm")
    else:
        model.load_model('../../../results/v2/chinese_gender/comparison/classification/_mean/mean_4_1.cbm', format="cbm")
    return model

def load_regressor(without_w=False):
    from catboost import CatBoostRegressor
    model = CatBoostRegressor()

    if not without_w:
        model.load_model('../../../results/v2/chinese_gender/comparison/regression/_mean/mean_50.cbm', format="cbm")
    else:
        model.load_model('../../../results/v2/chinese_gender/comparison/regression/_mean/mean_0.cbm', format="cbm")
    return model

# def predict(model):
#     # prepare features
#     col_features = model.feature_names_
#     column_name = 'givenname' if 'givenname' in df_ch_names.columns else 'character' if 'character' in df_ch_names.columns else None
#     if column_name is None:
#         raise ValueError('Column "givenname" or "character" not found in dataframe')

#     # prepares inferred genders by the three toolsL W (infer on the fly if not present), N, R. For the latter two, they must be present in the dataframe.
#     # it requires: gender_w, NG, ngender_n
#     df_ch_names = tools_comparison.prepare_features(df_ch_names, column_name, resources_dir, cache_dir)
#     ios.printf(f'Features: {col_features} ({len(col_features)})')

#     missing_features = set(col_features) - set(df_ch_names.columns)
#     if missing_features:
#         raise ValueError(f'Missing features in dataframe: {missing_features}')

#     ###################
#     # Predict
#     ###################
#     print(df_ch_names.head(2))
#     # predict and decode (from number to label)
#     if is_classifier:
#         unknown_id = len(setup.LABELS)
#         df_ch_names.loc[:,'pred_code'] = df_ch_names.apply(lambda row: unknown_id if ios.pd.isna(row.ngender_r) or ios.pd.isna(row.ngender_w) else model.predict(row[col_features])[0], axis=1)
#         df_ch_names.loc[:,'pred_label'] = df_ch_names.apply(lambda row: categorical.UNKNOWN if row.pred_code == unknown_id else setup.LABELS[row.pred_code], axis=1)
#     else:
#         df_ch_names.loc[:,'pred'] = df_ch_names.apply(lambda row: ios.pd.NA if ios.pd.isna(row.ngender_r) or ios.pd.isna(row.ngender_w) else model.predict(row[col_features]), axis=1)
#     print(df_ch_names.sample(2))

#     # Save
#     fname = f"{'class' if is_classifier else 'reg'}_{ios.os.path.basename(model_fn).split('.')[0]}_{ios.os.path.basename(input_fn)}"
#     fn = ios.path_join(output_dir, fname)
#     ios.to_csv(df_ch_names, fn)
#     ios.printf(f'Saved: {fn}')

In [ ]:
for fn in glob.glob(path_predictions + '/*.csv'):
    print('\n', fn)

    # 1. Load data
    df_pred = pd.read_csv(fn, index_col=0) # predictions
    print(f"names: {df_pred.shape} (expected)")

    # 2. Load genders per tool
    df_r = pd.read_csv(path_gender_tools.replace('<tool>', 'r')).rename(columns={'NG':'ngender_r'})[['givenname','ngender_r']]
    df_w = pd.read_csv(path_gender_tools.replace('<tool>', 'w')).rename(columns={'gender':'gender_w'})[['givenname','gender_w','pinyin','english']]
    df_n = pd.read_csv(path_gender_tools.replace('<tool>', 'n')).rename(columns={'likelyGender':'gender_n', 'genderScale':'ngender_n'})[['givenname','gender_n','ngender_n']]
    print(f"tools: {df_r.shape} (r), {df_w.shape} (w), {df_n.shape} (n)")

    # 3. Conver scores to 0-1
    # @TODO

    # 4. Join data (tool predictions)
    df_pred_new = df_pred.set_index('givenname').join(df_names.set_index('givenname'), how='inner') # join with names

    # 5. missing in the new df, but needed:
    print('\nMissing but needed:')
    new_ids = df_names.query("givenname not in @df_pred_new.index").index
    print(df_names.loc[new_ids].givenname.tolist())
    
    df_pred_new = pd.concat([df_pred_new, df_names.loc[new_ids,['givenname','pinyin','english']].set_index('givenname')], axis=0) 

    print(f"final df: {df_pred_new.shape} (observed)")

    # # 6. Join with tool predictions
    # df_pred_new = df_pred_new.join(df_r.set_index('givenname'), how='inner', rsuffix='_r')
    # df_pred_new = df_pred_new.join(df_w.set_index('givenname'), how='inner', rsuffix='_w')
    # df_pred_new = df_pred_new.join(df_n.set_index('givenname'), how='inner', rsuffix='_n')
    

    

    # # 7. prediction on new names:
    # for id, row in df_names.loc[new_ids].iterrows():
    #     print(id, row.givenname, row.pinyin, row.english)

    # 8. missing in the new df, but not needed:
    print('\nMissing but not needed:')
    print(df_pred.query("givenname not in @df_pred_new.index").givenname.tolist())


    break


 ../../results/v2/chinese_gender/predictions/class_mean_230_1_CNKI_uniqueChineseGivenNames_gender.csv
names: (39713, 43) (expected)
tools: (39714, 2) (r), (39714, 4) (w), (39714, 3) (n)

Missing but needed:
['屿']
final df: (39713, 44) (observed)

Missing but not needed:
['格尔']


In [54]:
df_pred_new

,ngender_r,gender_w,0,1,2,3,4,5,6,7,...,mode_ngender_rnw,mode_ngender_rn,mode_ngender_rw,mode_ngender_nw,gender_nw,pred_code,pred_label,pinyin,english,ngender_r_r
givenname,,,,,,,,,,,,,,,,,,,,,
俊池,0.30315,male,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.30315,0.30315,0.30315,0.373277,male,0.0,male,jùnchí,junchi,0.3937
建军,0.14830,male,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.14830,0.14830,0.14830,0.307495,male,0.0,male,jiànjūn,jianjun,0.7034
茂友,0.22265,male,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.22265,0.22265,0.22265,0.328072,male,0.0,male,màoyǒu,maoyou,0.5547
传昆,0.21170,male,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.21170,0.21170,0.21170,0.312287,male,0.0,male,chuánkūn#####zhuànkūn,chuankun#####zhuankun,0.5766
建华,0.35810,female,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.35810,0.35810,0.35810,0.374521,neutral,0.0,male,jiànhuá#####jiànhuà#####jiànhuā,jianhua,0.2838
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
德路,0.27785,male,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.27785,0.27785,0.27785,0.345120,male,0.0,male,délù,delu,0.4443
彩平,0.67675,male,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.67675,0.67675,0.67675,0.351355,male,2.0,female,cǎipíng,caiping,-0.3535
良瑛,0.51740,female,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.51740,0.51740,0.51740,0.464830,neutral,2.0,female,liángyīng,liangying,-0.0348
